In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from torchvision.datasets import MNIST
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, pairwise_distances

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


# 1. Data Loading and Preprocessing

In [ ]:
train_set = MNIST('./data', train=True, download=True)
test_set = MNIST('./data', train=False, download=True)

X_train = train_set.data.numpy().reshape(-1, 28*28) / 255.0
y_train = train_set.targets.numpy()

X_test = test_set.data.numpy().reshape(-1, 28*28) / 255.0
y_test = test_set.targets.numpy()

print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")

Train shape: (60000, 784)
Test shape: (10000, 784)


# 2. Baseline Model (Random Forest)

In [ ]:
rf_baseline = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_baseline.fit(X_train, y_train)

y_pred = rf_baseline.predict(X_test)
print("--- Classification Report (Baseline) ---")
print(classification_report(y_test, y_pred))
print(f"Overall Accuracy: {accuracy_score(y_test, y_pred):.4f}")

--- Classification Report (Baseline) ---
              precision    recall  f1-score   support

           0       0.97      0.99      0.98       980
           1       0.99      0.99      0.99      1135
           2       0.96      0.97      0.97      1032
           3       0.96      0.96      0.96      1010
           4       0.97      0.97      0.97       982
           5       0.98      0.96      0.97       892
           6       0.98      0.98      0.98       958
           7       0.97      0.96      0.97      1028
           8       0.96      0.95      0.96       974
           9       0.96      0.95      0.96      1009

    accuracy                           0.97     10000
   macro avg       0.97      0.97      0.97     10000
weighted avg       0.97      0.97      0.97     10000

Overall Accuracy: 0.9704


# 3. Active Learning Strategy Components

In [ ]:
def get_uncertainty_samples(model, X_pool, n_instances=10):
    probs = model.predict_proba(X_pool)
    max_probs = np.max(probs, axis=1)
    uncertain_indices = np.argsort(max_probs)[:n_instances]
    return uncertain_indices

In [ ]:
def get_diversity_samples(X_labeled, X_pool, n_instances=10, dev=device):
    X_pool_t = torch.tensor(X_pool, dtype=torch.float32).to(dev)
    X_labeled_t = torch.tensor(X_labeled, dtype=torch.float32).to(dev)

    distances_t = torch.cdist(X_pool_t, X_labeled_t, p=2)

    min_distances_t = torch.min(distances_t, dim=1)[0]

    min_distances = min_distances_t.cpu().numpy()
    diverse_indices = np.argsort(min_distances)[-n_instances:]

    return diverse_indices

In [ ]:
def get_combined_samples(model, X_labeled, X_pool, n_instances=10):
  uncertain_candidates_idx = get_uncertainty_samples(model=model, X_pool=X_pool, n_instances=n_instances)
  X_candidates = X_pool[uncertain_candidates_idx]

  diverse_sub_idx = get_diversity_samples(X_labeled=X_labeled, X_pool=X_candidates, n_instances=n_instances)

  return uncertain_candidates_idx[diverse_sub_idx]